### Ingestion del archivo "movie.csv"

Ejecuto un notebook desde este notebook

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/commom_functions"

In [0]:
# bronze_folder_path

#### Paso 1 - Leer el archivo CSV usando "DataFrameReader" de Spark

#### ver esquema

desde la DataFrameReader se pueden agregar opciones adicionales para leer el esquema de forma predeterminada por spark. No es recomendable para datos grandes. Ya que hace dos jobs.. uno para leer y otro para asignar el schema.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

In [0]:
movieschema = StructType( fields= [
    StructField('movieId', IntegerType(), False),
    StructField('title', StringType(), True),
    StructField('budget', DoubleType(), True),
    StructField('homePage', StringType(), True),
    StructField('overview', StringType(), True), 
    StructField('popularity', DoubleType(), True),
    StructField('yearReleaseDate', IntegerType(), True),
    StructField('releaseDate',  DateType(), True),
    StructField('revenue', DoubleType(), True),
    StructField('durationTime', IntegerType(), True),
    StructField('movieStatus', StringType(), True),
    StructField('tagline', StringType(), True),
    StructField('voteAverage', DoubleType(), True),
    StructField('voteCount', IntegerType(), True)
])

In [0]:
movie_df = spark.read \
  .option("header", True) \
  .schema(movieschema) \
  .csv(f"{bronze_folder_path}/{v_file_date}/movie.csv")

In [0]:
# display(movie_df)


#### Paso 2 - Seleccionar SOLO las columnas "requeridas"

In [0]:
# 4ta forma
from pyspark.sql.functions import col
movies_selected_df = movie_df.select(col("movieId"), col("title"), col("budget"), col("popularity"), col("yearReleaseDate"), col("releaseDate"), col("revenue"), col("durationTime"), col("voteAverage"),col("voteCount"))

In [0]:
# display(movies_selected_df)

#### Paso 3 - Renombrar columnas

In [0]:
# # 2da forma
movies_renamed_df = movies_selected_df \
    .withColumnsRenamed({"movieId":"movie_id","durationTime":"duration_time","releaseDate":"release_date","yearReleaseDate":"year_release_date","voteAverage":"vote_average","voteCount":"vote_count"})

In [0]:

# display(movies_renamed_df)

#### Paso 4 - Agregar una columna "ingestion_date" al dataframe

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
# 2da forma
# movies_final_df = movies_renamed_df.withColumns({"ingestion_date":current_timestamp(), "environment":lit("production")})  

# Uso la reutilizacion de codigo para ingestion_date
movies_final_df = add_ingestion_date(movies_renamed_df) \
                    .withColumn("environment",lit(v_environment)) \
                    .withColumn("file_date",lit(v_file_date))

In [0]:
# display(movies_final_df)

#### Paso 5 - Escribir datos en el datalake en formato "Parquet"

In [0]:
# hago una validacion con merge

merge_condition = 'tgt.movie_id = src.movie_id AND tgt.file_date=src.file_date'
merge_delta_lake(movies_final_df, "movie_silver", "movies", merge_condition, "file_date")

In [0]:
%sql
SELECT file_date, COUNT(1)
FROM movie_silver.movies
GROUP BY file_date

In [0]:
%sql DESC EXTENDED movie_silver.movies

In [0]:
# %fs
# ls abfss://silver@moviehistory22.dfs.core.windows.net/movies

In [0]:
# df =  spark.read.parquet("abfss://silver@moviehistory22.dfs.core.windows.net/movies")

In [0]:
# display(df)

In [0]:
dbutils.notebook.exit("Success")